# 09.1 - What Is a Language Model?

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

A **language model (LM)** is a statistical system trained to predict the next token (word, subword, or character) given the preceding tokens. Modern LLMs are deep transformers trained on massive text corpora. They power chat, summarization, code generation, and more.

## 2. Why Does This Matter?

Everything else in generative AI builds on this idea: tokenization, embeddings, attention, inference, prompting, and tool calling are all parts of the same next-token prediction loop. If you do not understand what an LLM is doing mechanically, you cannot debug failures or design reliable systems.

## 3. Prerequisites

- Phase 08 (Transformers)
- Basic probability (conditional probability)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain next-token prediction and conditional probability
- Describe autoregressive generation (left to right)
- Distinguish generation from retrieval
- Generate text with a tiny model you can understand

## 5. Mental Model

An LLM is a very large **autocomplete engine**. Given "The cat sat on the", it assigns a probability to every possible next token (mat 0.4, floor 0.2, table 0.15, ...) and picks one. Repeating this generates entire responses.

```text
Text -> tokens -> embeddings -> transformer layers -> probability distribution -> sampled token
                                            ^                                      |
                                            +----------- next token loop -------------+
```


## 6. Setup

We use a tiny randomly-initialized GPT-2-style model (no downloaded weights) to keep everything fast and CPU-friendly.


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import numpy as np
import torch.nn.functional as F
torch.manual_seed(42)
np.random.seed(42)


## 7. Build a Tiny Language Model

A language model is a function: given a sequence of token IDs, return a probability distribution over the next token for every position.


In [2]:
from transformers import GPT2Config, GPT2LMHeadModel

VOCAB = 100          # tiny vocabulary size
config = GPT2Config(
    n_layer=2,       # 2 transformer blocks
    n_head=2,        # 2 attention heads
    n_embd=32,       # small hidden size
    vocab_size=VOCAB,
    n_positions=64,  # context length
)
model = GPT2LMHeadModel(config)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

# Feed a random sequence of 10 token IDs
x = torch.randint(0, VOCAB, (1, 10))
with torch.no_grad():
    out = model(x)
logits = out.logits          # [1, 10, vocab]
print("Logits shape:", tuple(logits.shape))

# Last position logits -> next-token distribution
next_logits = logits[:, -1, :]           # [1, vocab]
probs = F.softmax(next_logits, dim=-1)   # [1, vocab]
print("Sum of probabilities:", probs.sum().item())
print("Probability of top token:", probs.max().item())


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 99), got 50256. This may result in unexpected behavior.


[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 99), got 50256. This may result in unexpected behavior.


Parameters: 30,720
Logits shape: (1, 10, 100)
Sum of probabilities: 1.0
Probability of top token: 0.012876146472990513


## 8. The Autoregressive Generation Loop

Autoregressive generation: at each step, feed the sequence in, read the distribution over the next token, pick one, append it, repeat. This is what makes an LM *generate*. We implement the loop by hand.


In [3]:
def sample_token(distribution, temperature=1.0):
    dist = (distribution / temperature).softmax(dim=-1)
    return torch.multinomial(dist, 1).item()

def generate(model, start_ids, max_new=20, temperature=1.0):
    ids = list(start_ids)
    with torch.no_grad():
        for _ in range(max_new):
            inp = torch.tensor([ids[-config.n_positions:]])
            logits = model(inp).logits[:, -1, :]     # distribution for next token
            probs = F.softmax(logits / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            ids.append(nxt)
            if nxt == 0:  # treat 0 as STOP
                break
    return ids

start = [5, 8, 2]   # arbitrary 'The cat sat on the' substitute
seq = generate(model, start, max_new=15)
print("Generated token IDs:", seq)
print("Copies you would pass to a decoder in production.")


Generated token IDs: [5, 8, 2, 72, 81, 9, 17, 72, 8, 98, 5, 60, 61, 11, 84, 94, 45, 59]
Copies you would pass to a decoder in production.


## 9. Why Different Outputs Each Time

Sampling is stochastic: the same input can produce different outputs because we roll a weighted die at every step. Set a high temperature to flatten the distribution and see more variety.


In [4]:
runs = {}
for temp in [0.2, 1.0, 3.0]:
    out = [generate(model, start, max_new=8, temperature=temp) for _ in range(3)]
    runs[temp] = out
    print(f"temperature={temp}: {len(set(tuple(r) for r in out))} distinct outputs out of 3")
print()
print("Low temperature -> more deterministic; high temperature -> more varied.")


temperature=0.2: 3 distinct outputs out of 3
temperature=1.0: 3 distinct outputs out of 3


temperature=3.0: 3 distinct outputs out of 3

Low temperature -> more deterministic; high temperature -> more varied.


## 10. Generation vs Retrieval

A language model **generates** likely continuations. A search engine **retrieves** stored facts. The model does not 'look up' the answer; it predicts what text most plausibly follows. That is why it can be confident yet wrong (hallucination).

## 11. Failure Case

A tiny random model has no learned meaning, so its outputs are noise. In a real model, treating output as factual truth is the classic failure: it can produce fluent but false statements. Ground generation with retrieval or validation when facts matter.


In [5]:
# Demonstration: the model has no truth, only statistics
print("Tiny random model next-token top-3 at each of 3 positions:")
inp = torch.tensor([start])
with torch.no_grad():
    logits = model(inp).logits[0]
for i in range(len(start)):
    probs = logits[i].softmax(dim=-1)
    top3 = torch.topk(probs, 3)
    toks = [f"{p.item():.3f}" for p in top3.values]
    print(f"  after token id {start[i]}: {toks}")
print("\nThese probabilities come from random weights, not meaning - illustrating why")
print("outputs must be validated, not trusted.")


Tiny random model next-token top-3 at each of 3 positions:


  after token id 5: ['0.016', '0.012', '0.012']
  after token id 8: ['0.013', '0.013', '0.013']
  after token id 2: ['0.013', '0.013', '0.013']

These probabilities come from random weights, not meaning - illustrating why
outputs must be validated, not trusted.


## 12. Debugging: Common Errors

| Symptom | Likely Cause | Fix |
|---|---|---|
| Fluent but wrong output | Model predicts, does not retrieve | Ground with RAG / validate |
| Nonsense output | Tiny / untrained model or short input | Use a trained model, clearer context |
| Refuses answer | Alignment / content policy | Rephrase or use another model |
| Poor quality | Model too small for task | Bigger model or better prompting |

## 13. Real-World Considerations

- Training data cutoff: models cannot know events after their training date.
- Open-source vs proprietary models trade control / privacy against raw capability.
- Model size, context length, latency, and price are all real constraints.

## 14. Common Mistakes

- Treating output as factual retrieval.
- Ignoring the training data cutoff.
- Assuming the model 'understands' in a human sense.
- Using one model for every task.

## 15. When NOT to Use

- When you need deterministic, verifiable output only -> use retrieval or rules.
- When facts are critical and you cannot validate -> ground generation.

## 16. Challenge

Modify the `generate` function to track how the context changes between steps (print the sequence you feed in at each iteration) and confirm it grows by one token each loop.


In [6]:
def generate_with_trace(model, start_ids, max_new=5):
    ids = list(start_ids)
    with torch.no_grad():
        for step in range(max_new):
            inp = torch.tensor([ids[-config.n_positions:]])
            logits = model(inp).logits[:, -1, :]
            probs = logits.softmax(dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            print(f"  step {step}: context length {len(ids)} -> append token {nxt}, context now {len(ids)+1}")
            ids.append(nxt)
    return ids

generate_with_trace(model, start)
print("\nConfirmed: autoregressive generation grows the context by one token per step.")


  step 0: context length 3 -> append token 10, context now 4
  step 1: context length 4 -> append token 59, context now 5
  step 2: context length 5 -> append token 81, context now 6
  step 3: context length 6 -> append token 84, context now 7
  step 4: context length 7 -> append token 28, context now 8

Confirmed: autoregressive generation grows the context by one token per step.


## 17. Closed-Book Recall

Without looking back:

1. What is next-token prediction?
2. Why can an LLM produce different outputs for the same input?
3. What is the difference between a language model and a search engine?
4. Why does model size matter for generation quality?

## 18. Teach-Back Questions

Explain to another person:

- The autoregressive generation loop, step by step.
- Why generation is not the same as retrieval.

## 19. Summary

You built a tiny language model, implemented autoregressive generation by hand, explored temperature-driven variety, and learned why generation is not retrieval.

## 20. Further Experiment

- Train the tiny model on a small corpus (see Unit 09.5) and observe outputs improve.
- Implement greedy versus sampling decoding (see Unit 09.6).

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, transformers, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
